### Off-Policy Hyperparametertuning of the BADP Policy

Parameters to be tuned:
- Sample size in days
- discount-factor: theta
- weight of current energy amount: lambda

NOTE: We also consider multiple linear models for the approximation of the value function with and without regularization

In [1]:
import pandas as pd, numpy as np, plotly.express as px, plotly.graph_objects as go

In [2]:
from Models.EnergyStorageModel import EnergyStorageModel as ESM

In [3]:
import pandas as pd
import numpy as np


policy_train_files = {
    0.0: "./Data/policy_train.csv",
    0.15: "./Data/policy_train_all_feat_noise015.csv",
    0.25: "./Data/policy_train_all_feat_noise025.csv",
    0.35: "./Data/policy_train_all_feat_noise035.csv"
}

policy_test_files = {
    0.0: "./Data/policy_test.csv",
    0.15: "./Data/policy_test_all_feat_noise015.csv",
    0.25: "./Data/policy_test_all_feat_noise025.csv",
    0.35: "./Data/policy_test_all_feat_noise035.csv"
}

def load_and_reshape(file_path, keep_first_column=False):
    df = pd.read_csv(file_path)
    
    if not keep_first_column:
        df.drop(columns=["0"], inplace=True)
    
    array = df.to_numpy()
    num_cols = array.shape[1]
    new_length = (array.shape[0] // 24) * 24
    array = array[:new_length, :]
    reshaped_array = array.reshape(new_length // 24, 24, num_cols).transpose(2, 0, 1)
    
    return reshaped_array


In [9]:
from joblib import Parallel, delayed
from Models.BaseClasses.Util import grid_search_BADP
from Models.Policies.VFA import BADP

data = load_and_reshape(policy_train_files[0.0], keep_first_column=True)
train = data[0, :250, :] # train
test = data[0, 250:, :] # test

initial_state = {"energy_amount": 300, "price": test[0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": test[1:]}
T = len(test[1:])
t0 = 0

grid = {
    "sample_size": [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120],
    "discount_factor": [0.95, 0.99],
    "energy_bonus_factor": [0.08, 0.09],
}

model = ESM(
    model_name=model_name,
    S0=initial_state,
    init_args=init_args,
    exog_params=exog_params,
    t0=t0,
    T=T,
)

badp_policy = BADP(
    model=model,
    price_samples=train,
    sample_size=24,
    discount_factor=0.99,
    test_size=0.1,
    verbose=False,
    aggregation_method="mean",
    model_type="linear",
    energy_bonus_factor=0.07
)

result = grid_search_BADP(grid, badp_policy, n_iterations=1)
print("Best Parameters:", result["best_parameters"])
print("Best Performance:", result["best_performance"])
print("All Runs:\n", result["all_runs"])

result["all_runs"].to_csv("./Data/BADP_correct_hist_corr.csv", index=False)

Best Parameters: {'sample_size': 60, 'discount_factor': 0.99, 'energy_bonus_factor': 0.09}
Best Performance: 1277.1557934210527
All Runs:
     sample_size  discount_factor  energy_bonus_factor  performance
0             1             0.95                 0.08   992.556778
1             1             0.95                 0.09  1050.473375
2             1             0.99                 0.08  1035.226818
3             1             0.99                 0.09  1085.961014
4             2             0.95                 0.08  1057.432239
..          ...              ...                  ...          ...
67          110             0.99                 0.09  1266.215872
68          120             0.95                 0.08  1168.294903
69          120             0.95                 0.09  1249.565837
70          120             0.99                 0.08  1205.765942
71          120             0.99                 0.09  1269.929205

[72 rows x 4 columns]


In [11]:
from itertools import product

val_scen = load_and_reshape(policy_test_files[0.0], keep_first_column=True)
val = val_scen[0, :, :]

initial_state = {"energy_amount": 300, "price": test[0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": test[1:]}
T = len(test[1:])
t0 = 0

grid = {
    "sample_size": [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120],
    "discount_factor": [0.99],
    "energy_bonus_factor": [0.08, 0.09, 0.1],
}

val_model = ESM(
    model_name=model_name,
    S0={"energy_amount": 300, "price": val[0]},
    init_args=init_args,
    exog_params={"hist_price": val[1:]},
    t0=0,
    T=len(val[1:]),
)

val_policy = BADP(
    model=model,
    price_samples=train,
    sample_size=24,
    discount_factor=0.99,
    test_size=0.1,
    verbose=False,
    aggregation_method="combine",
    model_type="linear",
    energy_bonus_factor=0.07
)

results = []
for v in product(*grid.values()):
    params = dict(zip(grid.keys(), v))
    param_str = "_".join([f"{param}_{str(value).replace('.', '_')}" for param, value in params.items()])
    model_filename = f"/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_hist/model_{param_str}.pkl"
    scaler_filename = f"/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_hist/scaler_{param_str}.pkl"

    # Setze die Policy-Attribute basierend auf den Parametern
    badp_policy = BADP(
        model=val_model,
        price_samples=train,
        sample_size=params["sample_size"],
        discount_factor=params["discount_factor"],
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=params["energy_bonus_factor"]
    )

    # Lade das Modell und den Scaler
    try:
        badp_policy.load_model(model_filename, scaler_filename)
    except FileNotFoundError:
        print(f"Model or scaler file not found for parameters: {param_str}")
        continue

    # Test the policy
    performance = badp_policy.run_policy(n_iterations=1)
    params["performance"] = performance
    results.append(params)

# Speichere die Ergebnisse in einem DataFrame
results_df = pd.DataFrame(results)
print("All Runs:\n", results_df)

results_df.to_csv("./Data/BADP_correct_hist_test.csv", index=False)

All Runs:
     sample_size  discount_factor  energy_bonus_factor  performance
0             1             0.99                 0.08  7260.370982
1             1             0.99                 0.09  7149.531742
2             1             0.99                 0.10  6384.327937
3             2             0.99                 0.08  7771.206668
4             2             0.99                 0.09  7886.829062
5             2             0.99                 0.10  7154.526455
6             4             0.99                 0.08  7925.166907
7             4             0.99                 0.09  7882.333962
8             4             0.99                 0.10  6902.613864
9             6             0.99                 0.08  7981.600479
10            6             0.99                 0.09  7895.865528
11            6             0.99                 0.10  6937.719439
12            8             0.99                 0.08  8017.474186
13            8             0.99                 0.

In [4]:
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
from Models.BaseClasses.Util import grid_search_BADP
from Models.Policies.VFA import BADP

num_scenarios_list = [10]
noise_levels = [0.0]
all_results = []  # Liste für gesammelte Ergebnisse

def badp_training(noise, num_scenarios):
    scen_data = load_and_reshape(policy_train_files[noise])
    test_data = load_and_reshape(policy_train_files[0.0], keep_first_column=True)
    chosen_train_scenarios = np.random.choice(scen_data.shape[0], num_scenarios, replace=False)

    train_scen = [scen_data[idx, :250, :] for idx in chosen_train_scenarios]
    train_scen_stack = np.stack(train_scen, axis=0)
    test = test_data[0, 250:, :]

    initial_state = {"energy_amount": 300, "price": test[0]}
    init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
    model_name = "cnf-24"
    exog_params = {"hist_price": test[1:]}
    T = len(test[1:])
    t0 = 0

    grid = {
        "sample_size": [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120],
        "discount_factor": [0.99],
        "energy_bonus_factor": [0.09],
    }

    model = ESM(
        model_name=model_name,
        S0=initial_state,
        init_args=init_args,
        exog_params=exog_params,
        t0=t0,
        T=T,
    )

    badp_policy = BADP(
        model=model,
        price_samples=train_scen_stack,
        sample_size=24,
        discount_factor=0.99,
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=0.07
    )

    result = grid_search_BADP(grid, badp_policy, n_iterations=1)

    print(f"Noise: {noise}, Num Scenarios: {num_scenarios}")
    print("Best Parameters:", result["best_parameters"])
    print("Best Performance:", result["best_performance"])

    result_df = result["all_runs"].copy()
    result_df["noise"] = noise
    result_df["num_scenarios"] = num_scenarios

    return result_df  

all_results = Parallel(n_jobs=-1)(
    delayed(badp_training)(noise, num_scenarios) for noise in noise_levels for num_scenarios in num_scenarios_list
)

final_results_df = pd.concat(all_results, ignore_index=True)

final_results_df.to_csv("./Data/BADP_corr_scen_comb_train.csv", index=False)

Noise: 0.0, Num Scenarios: 10
Best Parameters: {'sample_size': 90, 'discount_factor': 0.99, 'energy_bonus_factor': 0.09}
Best Performance: 1276.8108223684205


In [5]:
from itertools import product

val_scen = load_and_reshape(policy_test_files[0.0], keep_first_column=True) 
val = val_scen[0, :, :]

initial_state = {"energy_amount": 300, "price": val[0]}
init_args = {"eta": 0.95, "Rmax": 600, "max_load_per_hour": 50}
model_name = "cnf-24"
exog_params = {"hist_price": val[1:]}
T = len(val[1:])
t0 = 0

grid = {
    "sample_size": [1, 2, 4, 6, 8, 10, 15, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120],
    "discount_factor": [0.99],
    "energy_bonus_factor": [0.09],
}

val_model = ESM(
    model_name=model_name,
    S0={"energy_amount": 300, "price": val[0]},
    init_args=init_args,
    exog_params={"hist_price": val[1:]},
    t0=0,
    T=len(val[1:]),
)

val_policy = BADP(
    model=val_model,
    price_samples=val,
    sample_size=24,
    discount_factor=0.99,
    test_size=0.1,
    verbose=False,
    aggregation_method="combine",
    model_type="linear",
    energy_bonus_factor=0.07
)

results = []
for v in product(*grid.values()):
    params = dict(zip(grid.keys(), v))
    param_str = "_".join([f"{param}_{str(value).replace('.', '_')}" for param, value in params.items()])
    model_filename = f"/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_comb/model_{param_str}.pkl"
    scaler_filename = f"/Users/florian/Documents/github/thesis/stochastic-optimization/EnergyStorage_II/vfa_models/correct_scen_comb/scaler_{param_str}.pkl"

    # Setze die Policy-Attribute basierend auf den Parametern
    badp_policy = BADP(
        model=val_model,
        price_samples=val,
        sample_size=params["sample_size"],
        discount_factor=params["discount_factor"],
        test_size=0.1,
        verbose=False,
        aggregation_method="mean",
        model_type="linear",
        energy_bonus_factor=params["energy_bonus_factor"]
    )

    # Lade das Modell und den Scaler
    try:
        badp_policy.load_model(model_filename, scaler_filename)
    except FileNotFoundError:
        print(f"Model or scaler file not found for parameters: {param_str}")
        continue

    # Test the policy
    performance = badp_policy.run_policy(n_iterations=1)
    params["performance"] = performance
    results.append(params)

# Speichere die Ergebnisse in einem DataFrame
results_df = pd.DataFrame(results)
print("All Runs:\n", results_df)

results_df.to_csv("./Data/BADP_correct_scen_comb_test.csv", index=False)

All Runs:
     sample_size  discount_factor  energy_bonus_factor  performance
0             1             0.99                 0.09  6497.605533
1             2             0.99                 0.09  7654.677368
2             4             0.99                 0.09  7785.279884
3             6             0.99                 0.09  7849.140753
4             8             0.99                 0.09  7851.930480
5            10             0.99                 0.09  7950.305413
6            15             0.99                 0.09  8012.643961
7            20             0.99                 0.09  8058.281828
8            30             0.99                 0.09  8032.991992
9            40             0.99                 0.09  8113.714825
10           50             0.99                 0.09  8101.318782
11           60             0.99                 0.09  8108.249838
12           70             0.99                 0.09  8145.978787
13           80             0.99                 0.